# Notebook 05 preparation and runtime calibration definitions

This is a **build-only definition module**. Running this module itself loads reusable functions and performs no dataset preparation, GPU computation, optimizer update or experiment launch. The Notebook 05 controller calls these tagged definitions only when the user runs that notebook.

Inputs are the pinned Food101 source, original encoded/decoded image audit, original split, stage specification and frozen model configuration. Outputs at user execution are immutable stage split/data/verification manifests, discarded calibration checkpoints, numerical certificates and a runtime estimate. Stage data preserves the original development and official test reservations. The data-volume stage expands pretraining only; the supervised probe pool remains10,100 images.

Runtime calibration measures three warmup plus five real-data updates per architecture, starting at microbatch128 and falling back only for OOM or a70GiB allocation cap. Effective batch remains128. The registered learning schedule remains1,000 updates; calibration weights are discarded. Expected calibration is a few minutes, but higher-resolution cost is unknown until measured. All calls accept the controller's single absolute deadline; this module creates no new time allowance.

Checkpoints: `checkpoints/<run_id>/stages/<stage_id>/calibration/<case>/`. Resume by running Notebook05 top-to-bottom with exactly the same run ID, exported definitions, stage configuration and original deadline. Completed cases are verified/reused; interrupted cases resume from their full-state manifest. OOM cases remain explicit. Never reset a deadline or silently reduce the1,000-update learning budget.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import json
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa")
for cell in json.loads((ROOT / "fmm_jepa/bootstrap.ipynb").read_text())["cells"]:
    if cell["cell_type"] == "code" and "ahr-export" in cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(cell["source"]), "bootstrap#ahr-export", "exec"))
install_notebook_package(ROOT)


In [ ]:
from pathlib import Path
import copy, gc, hashlib, importlib.util, json, math, statistics, sys, time
import torch
from fmm_jepa.bootstrap import TaggedNotebookLoader
from fmm_jepa.food101_data import Food101Adapter, _read_sealed, canonical_hash
from fmm_jepa.semantic_pilot import configure_environment, initialize_models, run_model
from fmm_jepa.food101_training import validate_training_manifest
from fmm_jepa.training import atomic_json, definition_hashes, digest_file, run_writer
from fmm_jepa.geometry import clear_geometry_cache

PREPARATION_SCHEMA = "fmm-jepa-scaling-preparation/v1"
STAGE_DEFINITIONS = {
    "learning": {"id": "learning", "image_size": 128, "pretrain_pool": "selected"},
    "resolution": {"id": "resolution", "image_size": 256, "pretrain_pool": "selected"},
    "data_volume": {"id": "data_volume", "image_size": 128, "pretrain_pool": "full"},
}
CALIBRATION_KINDS = ("vit", "semantic")


def load05_module(root, name):
    """Load explicitly tagged definitions only; never execute notebook workflow cells."""
    if not isinstance(name, str) or not name or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in name):
        raise ValueError("Unsafe Notebook05 module name")
    path = Path(root) / "experiments/05_scaling" / (name + ".ipynb")
    key = "fmm05_" + name
    if key in sys.modules:
        module = sys.modules[key]
        if Path(module.__file__).resolve() != path.resolve():
            raise ImportError("Notebook05 module name already belongs to a different path")
        return module
    loader = TaggedNotebookLoader(path)
    spec = importlib.util.spec_from_loader(key, loader)
    module = importlib.util.module_from_spec(spec)
    sys.modules[key] = module
    try:
        loader.exec_module(module)
    except BaseException:
        sys.modules.pop(key, None)
        raise
    return module


def preparation_source_hashes(root):
    root = Path(root)
    hashes = dict(definition_hashes(root))
    for folder in ("experiments/04_followup", "experiments/05_scaling"):
        for path in sorted((root / folder).glob("*.ipynb")):
            notebook = json.loads(path.read_text())
            sources = ["".join(c["source"]) for c in notebook["cells"] if c["cell_type"] == "code"
                       and "ahr-export" in c.get("metadata", {}).get("tags", [])]
            if sources:
                hashes[str(path.relative_to(root))] = canonical_hash(sources)
    return hashes


def _immutable_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError("Immutable evidence differs; use a new run/stage identity: " + str(path))
    else:
        atomic_json(path, value)
    return value


def _stage_spec(stage):
    if isinstance(stage, str):
        if stage not in STAGE_DEFINITIONS:
            raise ValueError("Unknown Notebook05 stage")
        return copy.deepcopy(STAGE_DEFINITIONS[stage])
    result = dict(stage)
    if result.get("id") not in STAGE_DEFINITIONS or result != STAGE_DEFINITIONS[result["id"]]:
        raise ValueError("Stage differs from the registered learning/resolution/data-volume definitions")
    return result


def _stage_paths(root, artifact, stage):
    root, artifact = Path(root).resolve(), Path(artifact).resolve()
    stage = _stage_spec(stage)
    if artifact.name != stage["id"] or artifact.parent.name != "stages":
        raise ValueError("Expected artifacts/notebook_05/<run_id>/stages/<stage_id>")
    run_id = artifact.parent.parent.name
    expected = root / "artifacts/notebook_05" / run_id / "stages" / stage["id"]
    if artifact != expected:
        raise ValueError("Stage artifact escaped the Notebook05 run directory")
    if not run_id or any(not (c.isalnum() or c in "-_") for c in run_id):
        raise ValueError("Unsafe run ID")
    checkpoint = root / "checkpoints" / run_id / "stages" / stage["id"] / "calibration"
    return artifact, checkpoint, run_id


def derive_stage_memberships(source_split, stage):
    """Pure metadata operation. Probe IDs remain fixed when pretraining expands."""
    stage = _stage_spec(stage)
    selected = source_split["selected_ids"]
    pools = source_split["pool_ids"]
    train = list(selected["training"] if stage["pretrain_pool"] == "selected" else pools["training"])
    probe = list(selected["training"])
    dev = list(selected["development"])
    heldout = list(selected["heldout"])
    reserve = list(pools["development"])
    collections = {"training": train, "probe_training": probe, "development": dev,
                   "heldout": heldout, "development_reserve": reserve}
    for name, ids in collections.items():
        if len(ids) != len(set(ids)):
            raise ValueError("Duplicate sample IDs in " + name)
    expected_train = 10100 if stage["pretrain_pool"] == "selected" else 65600
    if len(train) != expected_train or len(probe) != 10100 or len(dev) != 2020:
        raise ValueError("Prior audited pool sizes differ from the registered Notebook05 study")
    if not set(probe).issubset(train) or not set(dev).issubset(reserve):
        raise ValueError("Fixed probe/development membership is not nested in the audited pools")
    if not set(train).issubset(pools["training"]):
        raise ValueError("New pretraining IDs escaped the prior audited training pool")
    if set(train) & set(reserve) or set(train) & set(heldout) or set(reserve) & set(heldout):
        raise ValueError("Training, full development reserve or official test overlap")
    if any(not sid.startswith("food101:train:") for sid in train + probe + dev):
        raise ValueError("Training/development IDs must originate in the official train split")
    if any(not sid.startswith("food101:test:") for sid in heldout):
        raise ValueError("Official test identity mismatch")
    return collections


def prepare_stage_bundle(root, artifact_dir, stage, config, verify_images=True):
    """At user execution, construct immutable stage metadata and verify selected source bytes.

    Original split, exclusions, labels, manifests and encoded images are unchanged.
    Full pretraining uses65,600 audited training identities; the probe always uses
    the exact original10,100 IDs. Only train/development image bytes are read.
    """
    root = Path(root).resolve()
    stage = _stage_spec(stage)
    artifact, _, run_id = _stage_paths(root, artifact_dir, stage)
    if int(config.get("image_size", stage["image_size"])) != stage["image_size"]:
        raise ValueError("Configuration image size differs from stage")
    if config.get("evaluation", {}).get("heldout_enabled", False):
        raise ValueError("Official test evaluation is outside Notebook05")
    source_run = config.get("source_food_run_id", "food101-pilot-001")
    if source_run != "food101-pilot-001":
        raise ValueError("Notebook05 preserves the original audited Food101 run")
    prior = root / "artifacts/notebook_02" / source_run
    dataset_doc = json.loads((prior / "dataset_manifest.json").read_text())
    split = json.loads((prior / "split_manifest.json").read_text())
    audit = json.loads((prior / "duplicate_audit.json").read_text())
    if canonical_hash({k:v for k,v in split.items() if k != "split_sha256"}) != split["split_sha256"]:
        raise ValueError("Original split checksum changed")
    if canonical_hash({k:v for k,v in audit.items() if k != "audit_sha256"}) != audit["audit_sha256"]:
        raise ValueError("Original decoded audit checksum changed")
    if split["duplicate_audit_sha256"] != audit["audit_sha256"]:
        raise ValueError("Original split and duplicate audit disagree")
    source = _read_sealed(root / "data/food101/dataset_manifest.json")
    fingerprints = {source["dataset_fingerprint"], dataset_doc["dataset_fingerprint"],
                    split["dataset_fingerprint"], audit["dataset_fingerprint"]}
    if len(fingerprints) != 1 or source["metadata_sha256"] != dataset_doc["metadata_sha256"]:
        raise ValueError("Original dataset and stage source metadata differ")
    memberships = derive_stage_memberships(split, stage)
    all_kept = set(memberships["training"]) | set(memberships["development_reserve"]) | set(memberships["heldout"])
    for group in audit["duplicate_groups"].values():
        if len(set(group) & all_kept) > 1:
            raise ValueError("Original exact-RGB duplicate exclusion failed in stage membership")
    excluded = {row["sample_id"] for row in split["exclusions"]}
    if all_kept & excluded:
        raise ValueError("A previously excluded original-image duplicate was reintroduced")
    dataset = Food101Adapter(root / "data/food101", source)
    selected = set(memberships["training"]) | set(memberships["probe_training"]) | set(memberships["development"])
    if not selected.issubset(dataset.index):
        raise ValueError("Stage selected IDs are absent from pinned source metadata")
    original_sources = {name: digest_file(prior / name) for name in
                        ("dataset_manifest.json", "split_manifest.json", "duplicate_audit.json")}
    derived = copy.deepcopy(split)
    derived.pop("split_sha256")
    derived.update(schema="fmm-jepa-scaling-stage-data/v1", stage=stage,
                   parent_split_sha256=split["split_sha256"],
                   selected_ids={name: memberships[name] for name in ("training", "probe_training", "development", "heldout")},
                   selected_sizes={name: len(memberships[name]) for name in ("training", "probe_training", "development", "heldout")},
                   selected_per_class={"training":100 if stage["pretrain_pool"] == "selected" else None,
                                       "probe_training":100, "development":20},
                   probe_training_policy="Fixed original10,100 selected training identities; never expanded with unlabeled pretraining pool.",
                   pretraining_pool=stage["pretrain_pool"], heldout_predictive_evaluation="NOT_RUN; reserved")
    derived["split_sha256"] = canonical_hash(derived)
    data_manifest = {"schema": PREPARATION_SCHEMA, "run_id": run_id, "stage": stage,
                     "dataset_fingerprint": dataset.fingerprint, "source_run": source_run,
                     "original_source_file_hashes": original_sources,
                     "original_split_sha256": split["split_sha256"],
                     "stage_split_sha256": derived["split_sha256"],
                     "membership_counts": {name: len(ids) for name,ids in memberships.items()},
                     "membership_hashes": {name: canonical_hash(ids) for name,ids in memberships.items()},
                     "sample_id_disjoint": True, "probe_training_subset_of_pretraining": True,
                     "reserved_development_pool_excluded_from_pretraining": True,
                     "exact_original_RGB_duplicate_exclusions_preserved": True,
                     "source_labels_unchanged": True, "pretraining_labels_enter_loss": False,
                     "test_pixels_read": 0, "heldout_status": "NOT_RUN",
                     "limitations": "Original exact decoded RGB exclusions only; near duplicates and semantic similarities are not ruled out."}
    data_manifest["identity_hash"] = canonical_hash(data_manifest)
    verification = {"status": "NOT_RUN", "verified_selected_images": 0, "test_pixels_read": 0}
    with run_writer(artifact / "preparation"):
        _immutable_json(artifact / "dataset_manifest.json", dataset_doc)
        _immutable_json(artifact / "split_manifest.json", derived)
        _immutable_json(artifact / "stage_data_manifest.json", data_manifest)
        if verify_images:
            verified = {}
            for entry in audit["original_record_shards"]:
                shard_path = root / "data/food101/audit_shards" / entry["file"]
                if digest_file(shard_path) != entry["sha256"]:
                    raise ValueError("Original encoded image audit shard changed")
                for record in json.loads(shard_path.read_text())["records"]:
                    sid = record["sample_id"]
                    if sid in selected:
                        path = dataset.path_for(sid)
                        if path.stat().st_size != record["encoded_bytes"] or digest_file(path) != record["encoded_sha256"]:
                            raise ValueError("Selected original image changed: " + sid)
                        verified[sid] = record["encoded_sha256"]
            if set(verified) != selected:
                raise ValueError("Some selected images lack original encoded-image audit records")
            verification = {"schema": PREPARATION_SCHEMA, "status": "PASS",
                            "stage_data_identity_hash": data_manifest["identity_hash"],
                            "verified_selected_images": len(verified),
                            "selected_encoded_hashes_sha256": canonical_hash(verified),
                            "original_decoded_audit_sha256": audit["audit_sha256"],
                            "verification": "Every selected train/probe/development encoded byte hash matches its original decoded-RGB audit record.",
                            "test_pixels_read": 0, "heldout_status": "NOT_RUN"}
            _immutable_json(artifact / "image_verification.json", verification)
    return {"dataset": dataset, "dataset_manifest": dataset_doc, "manifest": dataset_doc,
            "split_manifest": derived, "audit": audit, "fingerprint": dataset.fingerprint,
            "train_ids": memberships["training"], "probe_train_ids": memberships["probe_training"],
            "dev_ids": memberships["development"], "heldout_ids": memberships["heldout"],
            "diagnostic_ids": list(split["diagnostic_ids"]), "stage_data_manifest": data_manifest,
            "image_verification": verification, "stage": stage}


def _release_calibration(device):
    clear_geometry_cache()
    gc.collect()
    if str(device).startswith("cuda"):
        torch.cuda.synchronize(device)
        torch.cuda.empty_cache()


def _calibration_trace_hash(traces):
    names = ("attempted_update", "sample_ids", "occurrences", "augmentation_ids", "mask_ids", "successful", "successful_update")
    return canonical_hash([{name: trace[name] for name in names} for trace in traces])


def _load04_engineering(root):
    path = Path(root) / "experiments/04_followup/engineering.ipynb"
    loader = TaggedNotebookLoader(path)
    spec = importlib.util.spec_from_loader("scaling05_frozen04_engineering", loader)
    module = importlib.util.module_from_spec(spec)
    loader.exec_module(module)
    return module


def calibrate_stage(root, artifact_dir, stage, config, bundle, deadline_unix):
    """USER EXECUTION ONLY: discarded real-data updates; no automatic budget reduction.

    Try128 then64/32/16 only if either architecture OOMs or exceeds70GiB. Pick the
    largest common feasible microbatch. The full learning schedule remains1,000;
    stop_after=8 bounds calibration alone. Full-state checkpoints make partial
    calibration resumable. Training checks the original deadline at update
    boundaries; the accumulation certificate uses that same absolute deadline.
    The controller's persistent external watchdog remains authoritative.
    """
    root = Path(root).resolve()
    stage = _stage_spec(stage)
    artifact, checkpoint_root, run_id = _stage_paths(root, artifact_dir, stage)
    cfg = copy.deepcopy(config)
    if int(cfg.get("image_size", 0)) != stage["image_size"] or bundle["stage"] != stage:
        raise ValueError("Calibration stage/config/data identity mismatch")
    if bundle.get("image_verification", {}).get("status") != "PASS":
        raise ValueError("All selected original image bytes must verify before calibration")
    if int(cfg["training"]["updates"]) != 1000:
        raise ValueError("Notebook05 must retain the registered1,000-update schedule")
    cfg["training"]["checkpoint_every"] = 4
    cfg["training"].update(microbatch=128, accumulation_steps=1)
    cfg = configure_environment(cfg)
    directory = artifact / "calibration"
    directory.mkdir(parents=True, exist_ok=True)
    identity = {"schema": PREPARATION_SCHEMA, "stage": stage, "config": cfg,
                "source_hashes": preparation_source_hashes(root),
                "stage_data_identity_hash": bundle["stage_data_manifest"]["identity_hash"],
                "stage_split_sha256": bundle["split_manifest"]["split_sha256"],
                "microbatch_candidates": [128,64,32,16], "effective_batch": 128,
                "warmup_updates": 3, "measured_updates": 5,
                "peak_limit_bytes": 70*1024**3,
                "selection": "Largest common feasible microbatch; fallback only for OOM or allocation cap; no quality selection.",
                "learning_updates_unchanged":1000, "calibration_updates":8}
    identity_hash = canonical_hash(identity)
    _immutable_json(directory / "protocol.json", {"identity":identity,"identity_hash":identity_hash})
    final_path = artifact / "calibration_result.json"
    if final_path.exists():
        result = json.loads(final_path.read_text())
        if result["identity_hash"] != identity_hash or result["status"] != "PASS":
            raise ValueError("Completed calibration identity/status changed")
        for row in result["rows"]:
            if row["status"] == "PASS":
                validate_training_manifest(row["checkpoint_dir"])
                path = directory / row["case_key"] / "case.json"
                if json.loads(path.read_text()) != row:
                    raise ValueError("Calibration case evidence changed")
        certificate = directory / "gradient_certificate/microbatch_equivalence.json"
        if digest_file(certificate) != result["accumulation_certificate_sha256"]:
            raise ValueError("Accumulation certificate changed")
        return result
    if time.time() >= deadline_unix:
        raise TimeoutError("Original Notebook05 deadline has expired")
    started = time.time()
    initial_models, _ = initialize_models(cfg, directory)
    records = []
    common_trace = None
    selected = None
    try:
        for microbatch in identity["microbatch_candidates"]:
            feasible = True
            current = []
            for kind in CALIBRATION_KINDS:
                key = f"{kind}_mb{microbatch}"
                folder = directory / key
                folder.mkdir(exist_ok=True)
                path = folder / "case.json"
                checkpoint = checkpoint_root / key
                row = json.loads(path.read_text()) if path.exists() else None
                if row is not None and row.get("identity_hash") != identity_hash:
                    raise ValueError("Existing calibration case identity differs")
                if row is None:
                    if time.time() >= deadline_unix - 30:
                        raise TimeoutError("Insufficient original deadline for calibration and certificate")
                    local = copy.deepcopy(cfg)
                    local["training"].update(microbatch=microbatch,accumulation_steps=128//microbatch)
                    row = {"case_key":key,"model_id":kind,"microbatch":microbatch,
                           "accumulation_steps":128//microbatch,"effective_batch":128,
                           "identity_hash":identity_hash,"image_size":stage["image_size"]}
                    model = result = None
                    try:
                        _release_calibration(cfg["device"])
                        model = copy.deepcopy(initial_models[kind])
                        protocol = {"source_hashes":identity["source_hashes"],"calibration_identity_hash":identity_hash,
                                    "purpose":"Notebook05 real-data calibration; weights discarded; learning schedule unchanged"}
                        result = run_model(kind,local,bundle,folder,checkpoint,protocol,model=model,
                                           stop_after=8,deadline_unix=deadline_unix-15)
                        if result["resume"]["final_step"] != 8:
                            raise TimeoutError("Calibration stopped before eight complete updates")
                        if len(result["traces"]) != 8 or any(not t["successful"] for t in result["traces"]):
                            raise AssertionError("Calibration requires eight finite successful updates")
                        measured = result["metrics"][3:8]
                        if len(measured) != 5 or result["resume"]["actual_image_exposures"] != 1024:
                            raise AssertionError("Calibration sample/timing count mismatch")
                        peak = int(result["performance"]["peak_cuda_allocated_bytes"])
                        row.update(status="PASS" if peak <= identity["peak_limit_bytes"] else "MEMORY_LIMIT",
                                   successful_updates=8,image_exposures=1024,warmup_updates=3,measured_updates=5,
                                   median_step_seconds=statistics.median(m["step_seconds"] for m in measured),
                                   max_step_seconds=max(m["step_seconds"] for m in measured),
                                   raw_step_seconds=[m["step_seconds"] for m in measured],
                                   median_compute_seconds=statistics.median(m["compute_seconds"] for m in measured),
                                   median_data_seconds=statistics.median(m["data_seconds"] for m in measured),
                                   peak_cuda_allocated_bytes=peak,checkpoint_dir=str(checkpoint),
                                   trace_sha256=_calibration_trace_hash(result["traces"]),
                                   checkpoint_successful_updates=8)
                        manifest = json.loads((checkpoint / "manifest.json").read_text())
                        manifest.update(resume_notebook="notebooks/05_Learning_and_Resolution_Scaling.ipynb",
                                        resume_cell="launch-staged-study",
                                        resume_instructions="Run Notebook05 with unchanged run ID/configuration/sources and original deadline; this calibration case resumes from its valid manifest.")
                        atomic_json(checkpoint / "manifest.json",manifest)
                    except torch.cuda.OutOfMemoryError as exc:
                        row.update(status="OOM",error=str(exc),weights_discarded=True)
                    except RuntimeError as exc:
                        if "out of memory" not in str(exc).lower():
                            raise
                        row.update(status="OOM",error=str(exc),weights_discarded=True)
                    finally:
                        model = result = None
                        _release_calibration(cfg["device"])
                    _immutable_json(path,row)
                if row["status"] in ("PASS","MEMORY_LIMIT"):
                    validate_training_manifest(row["checkpoint_dir"])
                    traces = json.loads((folder / (kind+"_training_traces.json")).read_text())
                    trace_hash = _calibration_trace_hash(traces)
                    if trace_hash != row["trace_sha256"]:
                        raise ValueError("Calibration saved traces changed")
                    if common_trace is None:
                        common_trace = trace_hash
                    elif common_trace != trace_hash:
                        raise AssertionError("Calibration images, occurrences, augmentation or masks differ across cases")
                current.append(row)
                records.append(row)
                feasible = feasible and row["status"] == "PASS"
                print(f"Calibration {stage['id']} {key}: {row['status']}",flush=True)
            if feasible:
                selected = microbatch
                break
    finally:
        initial_models.clear()
        _release_calibration(cfg["device"])
    if selected is None:
        atomic_json(artifact / "calibration_failure.json",{"identity_hash":identity_hash,"status":"NO_COMMON_FEASIBLE_BATCH","rows":records})
        raise RuntimeError("No common FP32 microbatch in128/64/32/16 fits the registered70GiB cap; change configuration in a new run, not learning updates")
    # The frozen04 helper works at arbitrary image size and selected training pool.
    # Its evidence is isolated here and bound to this full stage/config/source identity.
    certificate_dir = directory / "gradient_certificate"
    certificate_dir.mkdir(exist_ok=True)
    _immutable_json(certificate_dir / "binding.json",{"calibration_identity_hash":identity_hash,"microbatch":selected,
                    "effective_batch":128,"baseline_microbatch":16,"source_hashes":identity["source_hashes"]})
    engineering = _load04_engineering(root)
    certificate = engineering.certify_microbatch_equivalence(root,certificate_dir,cfg,bundle,selected,deadline_unix)
    if certificate["status"] != "PASS":
        raise AssertionError("Actual128-image accumulation certificate failed")
    result = {"schema":PREPARATION_SCHEMA,"status":"PASS","identity":identity,"identity_hash":identity_hash,
              "rows":records,"selected_microbatch":selected,"selected_accumulation_steps":128//selected,
              "selected_model_step_seconds":{r["model_id"]:r["median_step_seconds"] for r in current},
              "selected_model_max_step_seconds":{r["model_id"]:r["max_step_seconds"] for r in current},
              "common_trace_sha256":common_trace,"actual_image_exposures_per_case":1024,
              "accumulation_certificate":certificate,"equivalence_status":certificate["status"],
              "accumulation_certificate_sha256":digest_file(certificate_dir / "microbatch_equivalence.json"),
              "wall_seconds":time.time()-started,"weights_discarded":True,"quality_evidence":False,
              "heldout_status":"NOT_RUN","registered_learning_updates":1000,
              "timing_scope":"Actual complete accumulated FP32 JEPA steps including real-data preparation/transfers, three warmups then five measured; checkpoint I/O excluded from per-step timing."}
    result["budget_estimate"] = estimate_stage_budget(cfg,result,remaining_seconds=float(deadline_unix)-time.time())
    result["fits_budget"] = result["budget_estimate"]["fits_budget"]
    _immutable_json(final_path,result)
    return result


def estimate_stage_budget(config, calibration=None, remaining_seconds=None, seed_count=3):
    """Transparent forecast only; never changes updates, seeds, data or probe budgets.

    The combined evaluation/preflight/report reserve is1,800 seconds at128
    resolution and3,600 at256, including18 fixed-final-epoch probe jobs. Neither
    reserve is an observed universal upper bound. Preparation/calibration reserve is300 seconds. Higher-resolution
    training must use measured calibration before the controller starts it.
    """
    updates = int(config["training"]["updates"])
    if updates != 1000 or int(seed_count) != 3:
        raise ValueError("This study registers1,000 updates and three paired seeds")
    image_size = int(config["image_size"])
    if calibration is None:
        if image_size != 128:
            return {"status":"UNKNOWN_REQUIRES_CALIBRATION","image_size":image_size,
                    "registered_updates":updates,"paired_seeds":3,"probe_jobs":18,
                    "fits_budget":None,"silent_budget_reduction":False,
                    "reason":"Resolution256 cost has not been measured; do not infer it from token count alone."}
        seconds = {"vit":1.455,"semantic":1.676}
        source = "Historical Notebook04 actual effective-batch128 FP32 timings; forecast only, not a measurement of Notebook05."
    else:
        if calibration.get("status") != "PASS":
            raise ValueError("Budget forecast requires completed calibration")
        if int(calibration["identity"]["stage"]["image_size"]) != image_size:
            raise ValueError("Calibration resolution differs from planned stage")
        seconds = {kind:float(calibration["selected_model_step_seconds"][kind]) for kind in CALIBRATION_KINDS}
        source = "This stage's discarded real-data calibration median complete-step timings."
    if any(not math.isfinite(v) or v <= 0 for v in seconds.values()):
        raise ValueError("Invalid calibration timing")
    training = int(seed_count) * updates * sum(seconds.values())
    settings = config.get("runtime_estimate",{})
    margin = float(settings.get("training_margin",1.2))
    minimum_reserve = 3600 if image_size == 256 else 1800
    reserve = float(settings.get("evaluation_preflight_report_reserve_seconds",
                    config.get("budget",{}).get("evaluation_reserve_seconds",minimum_reserve)))
    preparation = float(settings.get("preparation_calibration_reserve_seconds",300))
    if margin < 1.2 or reserve < minimum_reserve or preparation < 300:
        raise ValueError("Runtime uncertainty/reserve assumptions may not be silently weakened")
    predicted = training*margin+reserve+preparation
    result = {"status":"MEASURED_CALIBRATION_FORECAST" if calibration is not None else "HISTORICAL_FORECAST",
              "image_size":image_size,"registered_updates":updates,"paired_seeds":int(seed_count),
              "architectures":list(CALIBRATION_KINDS),"probe_jobs":18,
              "probe_job_accounting":"Six fresh initialization controls plus12 trained milestone controls (300 and1,000 updates); unchanged10,100 probe-fit images and2,020 development images.",
              "complete_step_seconds":seconds,"timing_source":source,
              "unmargined_training_seconds":training,"training_margin":margin,
              "evaluation_preflight_report_reserve_seconds":reserve,
              "preparation_calibration_reserve_seconds":preparation,
              "estimated_total_seconds":predicted,"estimated_total_hours":predicted/3600,
              "remaining_seconds":float(remaining_seconds) if remaining_seconds is not None else None,
              "fits_budget":None if remaining_seconds is None else predicted <= float(remaining_seconds),
              "silent_budget_reduction":False,
              "uncertainty":"A forecast, not a completion guarantee. Calibration does not measure complete feature extraction/probe runtime; higher-resolution evaluation may exceed the combined reserve. Persistent hard deadline remains authoritative.",
              "if_budget_does_not_fit":"Stop before stage training with INCOMPLETE_BUDGET, preserve prior evidence and report the unchanged requested budget. A different run/configuration must be selected explicitly; do not truncate the learning horizon or reset the clock."}
    return result


def require_stage_budget(config, calibration, deadline_unix, artifact_dir=None):
    estimate = estimate_stage_budget(config,calibration,remaining_seconds=float(deadline_unix)-time.time())
    if artifact_dir is not None:
        atomic_json(Path(artifact_dir) / "runtime_estimate.json",estimate)
    if estimate["fits_budget"] is not True:
        raise TimeoutError("INCOMPLETE_BUDGET: registered1,000-update three-seed stage is forecast not to fit the original remaining clock; no learning budget was reduced")
    return estimate


## CPU-only build validation

The checks below inspect existing split metadata and mock runtime estimates. They do not prepare a study, read image bytes, construct models, initialize CUDA or call an optimizer.

In [ ]:
# CPU-only build checks: source metadata and arithmetic, never image decoding or model work.
source_split = json.loads((ROOT / "artifacts/notebook_02/food101-pilot-001/split_manifest.json").read_text())
build_checks = {}
for stage_name in STAGE_DEFINITIONS:
    members = derive_stage_memberships(source_split, stage_name)
    assert members["probe_training"] == source_split["selected_ids"]["training"]
    assert members["development"] == source_split["selected_ids"]["development"]
    build_checks[stage_name] = {name:len(ids) for name,ids in members.items()}
leaking_split = copy.deepcopy(source_split)
leaking_split["pool_ids"]["training"][0] = leaking_split["pool_ids"]["development"][0]
try:
    derive_stage_memberships(leaking_split, "data_volume")
except ValueError:
    build_checks["leaking_pool_rejected"] = True
else:
    raise AssertionError("Leaking training pool was accepted")
forecast_config = {"image_size":128,"training":{"updates":1000}}
forecast = estimate_stage_budget(forecast_config,remaining_seconds=14400)
assert abs(forecast["estimated_total_seconds"] - 13371.6) < 1e-6 and forecast["fits_budget"]
assert estimate_stage_budget(forecast_config,remaining_seconds=3600)["fits_budget"] is False
assert estimate_stage_budget({"image_size":256,"training":{"updates":1000}})["fits_budget"] is None
mock_calibration = {"status":"PASS","identity":{"stage":{"image_size":256}},
                    "selected_model_step_seconds":{"vit":3.0,"semantic":4.0}}
resolution_estimate = estimate_stage_budget({"image_size":256,"training":{"updates":1000},
    "budget":{"evaluation_reserve_seconds":3600}},mock_calibration,remaining_seconds=24*3600)
assert resolution_estimate["evaluation_preflight_report_reserve_seconds"] == 3600
try:
    estimate_stage_budget({"image_size":128,"training":{"updates":999}})
except ValueError:
    build_checks["changed_learning_horizon_rejected"] = True
else:
    raise AssertionError("A silently shortened learning horizon was accepted")
assert not torch.cuda.is_initialized(), "Build validation must not initialize CUDA"
build_checks.update(status="PASS",cuda_initialized=False,optimizer_calls=0,
                    forecast_128_hours=forecast["estimated_total_hours"],resolution_requires_measured_calibration=True)
build_checks
